# Subnational Data Merge Pipeline (Step-by-Step)

This notebook breaks down the merge process into individual cells for debugging.

In [238]:
import os
import logging
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from rasterstats import zonal_stats
import glob
import difflib
from shapely.geometry import Point
import itertools

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Base Data Directory
# User specified to use 'data' (relative to notebook root)
DATA_DIR = "data"

# Parameters
iso3_list = ['KEN', 'SOM', 'ETH']
boundary_dir = os.path.join(DATA_DIR, 'geoboundaries')

print("Environment ready.")

Environment ready.


## 1. Helper Functions

In [ ]:

def load_night_lights(data_dir):
    try:
        # Check both potential paths
        path1 = os.path.join(data_dir, "processed/night_lights_admin2.parquet")
        path2 = os.path.join(data_dir, "night_lights/night_lights_admin2.parquet")
        
        path = path1 if os.path.exists(path1) else path2
        
        if os.path.exists(path):
            logger.info(f"Loading Night Lights from {path}...")
            nl_df = pd.read_parquet(path)
            # Rename for consistency if needed
            # Schema: country_iso, admin2, year, night_light_mean
            # We want to merge on [country_iso, admin2, year]
            # Rename admin2 -> admin2_canonical to match other merges if necessary
            # But our master skeleton uses 'admin2'. 
            # 'admin_gdf' has 'shapeName'.
            # Let's trust the names match since they come from the same GeoJSONs.
            logger.info(f"Night Lights loaded. Shape: {nl_df.shape}")
            return nl_df
        else:
            logger.warning("Night Lights parquet file not found.")
            return pd.DataFrame()
    except Exception as e:
        logger.error(f"Failed to load Night Lights: {e}")
        return pd.DataFrame()



In [239]:
# --- UPDATED SPI PROCESS FUNCTION (FIXED) ---
def process_spi_data(gdf, data_dir, start_year=2000):
    import xarray as xr
    import rioxarray
    from rasterstats import zonal_stats
    import numpy as np
    import pandas as pd
    from pathlib import Path
    
    spi_path = Path(data_dir) / 'processed' / 'spi' / 'spi_gamma_month6_spi.nc'
    if not spi_path.exists():
        logger.warning(f"SPI file not found: {spi_path}")
        return pd.DataFrame()
        
    try:
        ds = xr.open_dataset(spi_path)
        logger.info(f"Loaded SPI NetCDF. Filtering start_year >= {start_year}")
        
        # --- FIXES FOR RASTERSTATS ---
        # 1. Rename dimensions to x, y if strictly lon, lat
        if 'lon' in ds.dims and 'lat' in ds.dims:
            ds = ds.rename({'lon': 'x', 'lat': 'y'})
            
        # 2. Sort Y descending (Top-Down orientation)
        ds = ds.sortby('y', ascending=False)
        
        # 3. Ensure CRS
        if 'spatial_ref' not in ds.variables:
            ds = ds.rio.write_crs("EPSG:4326")
        # -----------------------------
            
        spi_var = 'spi_gamma_6_month'
        if spi_var not in ds.data_vars:
            spi_var = list(ds.data_vars)[0]
            
        times = ds.time.values
        results = []
        
        logger.info(f"Processing {len(times)} time slices for SPI...")
        
        for t_idx, t_val in enumerate(times):
            # Convert numpy.datetime64 to pd.Timestamp for easier year/month access
            ts = pd.Timestamp(t_val)
            year = ts.year
            month = ts.month
            
            # Start Year Filter
            if year < start_year:
                continue
            
            if t_idx % 24 == 0:
                logger.info(f"Processing SPI for {year}-{month}...")
                
            slice_da = ds[spi_var].isel(time=t_idx)
            arr = slice_da.values
            
            # Skip empty slices (e.g. 1991 start)
            if np.isnan(arr).all():
                continue
                
            transform = slice_da.rio.transform()
            
            # Zonal Stats with all_touched=True and nodata handling
            stats = zonal_stats(gdf, arr, affine=transform, stats="mean", nodata=np.nan, all_touched=True)
            
            for i, region in enumerate(gdf.itertuples()):
                mean_val = stats[i]['mean']
                country = getattr(region, 'country_iso', getattr(region, 'shapeISO', 'Unknown'))
                
                if mean_val is not None and not np.isnan(mean_val):
                    results.append({
                        'year': year,
                        'month': month,
                        'country_iso': country,
                        'admin1': getattr(region, 'admin1', 'Unknown'),
                        'admin2_canonical': getattr(region, 'shapeName', 'Unknown'),
                        'spi_6month': mean_val
                    })
                    
        df_spi = pd.DataFrame(results)
        logger.info(f"SPI Processing Complete. Rows: {len(df_spi)}")
        return df_spi
        
    except Exception as e:
        logger.error(f"Failed to process SPI: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()



In [240]:
def load_canonical_boundaries(iso3_list, boundary_dir):
    gdfs = []
    
    file_map = {
        'KEN': 'gb_KEN_ADM2.geojson',
        'SOM': 'gb_SOM_ADM2.geojson',
        'ETH': 'gb_ETH_ADM2.geojson'
    }
    
    for iso in iso3_list:
        filename = file_map.get(iso)
        path = os.path.join(boundary_dir, filename)
        
        if os.path.exists(path):
            try:
                # 1. Load Admin2 (Target Granularity)
                gdf = gpd.read_file(path)
                
                # Standardize Admin2 Name
                rename_map = {
                    'shapeName_ADM2': 'shapeName',
                    'ADM2_NAME': 'shapeName',
                    'admin2Name': 'shapeName',
                    'shapeName': 'shapeName' 
                }
                gdf.rename(columns=rename_map, inplace=True)
                
                # FIX: Force Populate shapeISO from loop variable
                # GeoJSONs have empty shapeISO, which breaks merges
                gdf['shapeISO'] = iso
                
                if 'shapeName' not in gdf.columns:
                    logger.warning(f"Could not identify ADM2 name column for {iso}. Available: {gdf.columns.tolist()}")
                    continue

                # 2. Get Admin1 info (Dynamic Spatial Join)
                # Check for sibling ADM1 file
                adm1_filename = filename.replace('ADM2', 'ADM1')
                adm1_path = os.path.join(boundary_dir, adm1_filename)
                
                if os.path.exists(adm1_path):
                    try:
                        adm1_gdf = gpd.read_file(adm1_path)
                         # Standardize Admin1 Name
                        adm1_rename = {
                            'shapeName': 'admin1',
                            'shapeName_ADM1': 'admin1',
                            'ADM1_NAME': 'admin1'
                        }
                        adm1_gdf.rename(columns=adm1_rename, inplace=True)
                        
                        if 'admin1' in adm1_gdf.columns:
                            # Spatial Join to get Admin1
                            # Ensure CRS match
                            if gdf.crs != adm1_gdf.crs:
                                adm1_gdf = adm1_gdf.to_crs(gdf.crs)
                                
                            # Point on Surface for join guarantees inside
                            # But sjoin works with polygons too.
                            # "inner" or "left" - we want for each ADM2, find its ADM1.
                            # It's better to use centroids if ADM2 boundaries are not perfectly inside ADM1
                            
                            # Use creating centroids for join
                            # But we need to keep geometry of ADM2
                            
                            # Let's use sjoin directly
                            joined = gpd.sjoin(gdf, adm1_gdf[['admin1', 'geometry']], how='left', predicate='intersects')
                            
                            # Handle duplicates if an ADM2 overlaps multiple ADM1s (take mode or first)
                            # Usually taking the one with largest intersection is best, but 'intersects' is binary.
                            # Let's assume clean hierarchy or take first.
                            
                            joined = joined.drop_duplicates(subset=['shapeName'])
                            
                            if 'admin1' in joined.columns:
                                gdf['admin1'] = joined['admin1']
                                logger.info(f"[{iso}] Spatially joined Admin1 info.")
                            else:
                                logger.warning(f"[{iso}] Spatial join failed to add admin1.")
                        else:
                             logger.warning(f"[{iso}] Admin1 file found but no name column.")
                    except Exception as e:
                        logger.warning(f"[{iso}] Failed to process Admin1 file: {e}")
                else:
                    logger.warning(f"[{iso}] No Admin1 reference file found at {adm1_path}")

                
                gdfs.append(gdf)
                logger.info(f"Loaded {iso} boundaries. Rows: {len(gdf)}")
                
            except Exception as e:
                logger.error(f"Failed to load {iso}: {e}")
                
    if not gdfs:
        return gpd.GeoDataFrame()
        
    full_gdf = pd.concat(gdfs, ignore_index=True)
    return full_gdf



## 2. Load Raw Data

In [222]:
print("Loading raw datasets...")
price_df = load_price_data(DATA_DIR)
crop_df = load_crop_data(DATA_DIR)
acled_df = load_acled_data(DATA_DIR)
macro_df = load_macro_data(DATA_DIR)

print(f"Price Rows: {len(price_df)}")
print(f"Crop Rows: {len(crop_df)}")
print(f"ACLED Rows: {len(acled_df)}")
print(f"Macro Rows: {len(macro_df)}")


# Load Night Lights
night_lights_df = load_night_lights(DATA_DIR)


Loading raw datasets...
Price Rows: 62560
Crop Rows: 1656
ACLED Rows: 48606
Macro Rows: 791


## 3. Load Canonical Boundaries

In [223]:
def load_canonical_boundaries(iso3_list, boundary_dir):
    gdfs = []
    
    file_map = {
        'KEN': 'gb_KEN_ADM2.geojson',
        'SOM': 'gb_SOM_ADM2.geojson',
        'ETH': 'gb_ETH_ADM2.geojson'
    }
    
    for iso in iso3_list:
        filename = file_map.get(iso)
        path = os.path.join(boundary_dir, filename)
        
        if os.path.exists(path):
            try:
                # 1. Load Admin2 (Target Granularity)
                gdf = gpd.read_file(path)
                
                # Standardize Admin2 Name
                rename_map = {
                    'shapeName_ADM2': 'shapeName',
                    'ADM2_NAME': 'shapeName',
                    'admin2Name': 'shapeName',
                    'shapeName': 'shapeName' 
                }
                gdf.rename(columns=rename_map, inplace=True)
                
                # FIX: Force Populate shapeISO from loop variable
                # GeoJSONs have empty shapeISO, which breaks merges
                gdf['shapeISO'] = iso
                
                if 'shapeName' not in gdf.columns:
                    logger.warning(f"Could not identify ADM2 name column for {iso}. Available: {gdf.columns.tolist()}")
                    continue

                # 2. Get Admin1 info (Dynamic Spatial Join)
                # Check for sibling ADM1 file
                adm1_filename = filename.replace('ADM2', 'ADM1')
                adm1_path = os.path.join(boundary_dir, adm1_filename)
                
                if os.path.exists(adm1_path):
                    try:
                        adm1_gdf = gpd.read_file(adm1_path)
                         # Standardize Admin1 Name
                        adm1_rename = {
                            'shapeName': 'admin1',
                            'shapeName_ADM1': 'admin1',
                            'ADM1_NAME': 'admin1'
                        }
                        adm1_gdf.rename(columns=adm1_rename, inplace=True)
                        
                        if 'admin1' in adm1_gdf.columns:
                            # Spatial Join to get Admin1
                            # Ensure CRS match
                            if gdf.crs != adm1_gdf.crs:
                                adm1_gdf = adm1_gdf.to_crs(gdf.crs)
                                
                            # Point on Surface for join guarantees inside
                            # But sjoin works with polygons too.
                            # "inner" or "left" - we want for each ADM2, find its ADM1.
                            # It's better to use centroids if ADM2 boundaries are not perfectly inside ADM1
                            
                            # Use creating centroids for join
                            # But we need to keep geometry of ADM2
                            
                            # Let's use sjoin directly
                            joined = gpd.sjoin(gdf, adm1_gdf[['admin1', 'geometry']], how='left', predicate='intersects')
                            
                            # Handle duplicates if an ADM2 overlaps multiple ADM1s (take mode or first)
                            # Usually taking the one with largest intersection is best, but 'intersects' is binary.
                            # Let's assume clean hierarchy or take first.
                            
                            joined = joined.drop_duplicates(subset=['shapeName'])
                            
                            if 'admin1' in joined.columns:
                                gdf['admin1'] = joined['admin1']
                                logger.info(f"[{iso}] Spatially joined Admin1 info.")
                            else:
                                logger.warning(f"[{iso}] Spatial join failed to add admin1.")
                        else:
                             logger.warning(f"[{iso}] Admin1 file found but no name column.")
                    except Exception as e:
                        logger.warning(f"[{iso}] Failed to process Admin1 file: {e}")
                else:
                    logger.warning(f"[{iso}] No Admin1 reference file found at {adm1_path}")

                
                gdfs.append(gdf)
                logger.info(f"Loaded {iso} boundaries. Rows: {len(gdf)}")
                
            except Exception as e:
                logger.error(f"Failed to load {iso}: {e}")
                
    if not gdfs:
        return gpd.GeoDataFrame()
        
    full_gdf = pd.concat(gdfs, ignore_index=True)
    return full_gdf



In [224]:
logger.info(f"Loading canonical boundaries from {boundary_dir}...")
admin_gdf = load_canonical_boundaries(iso3_list, boundary_dir)
if not admin_gdf.empty:
    canonical_names = admin_gdf['shapeName'].unique().tolist()
    print(f"Loaded {len(admin_gdf)} total regions.")
    print(admin_gdf.head())
else:
    print("No boundaries loaded! Check file paths.")

2026-02-12 14:11:41,773 - INFO - Loading canonical boundaries from data/geoboundaries...
2026-02-12 14:11:42,140 - INFO - [KEN] Spatially joined Admin1 info.
2026-02-12 14:11:42,141 - INFO - Loaded KEN boundaries. Rows: 290
2026-02-12 14:11:42,196 - INFO - [SOM] Spatially joined Admin1 info.
2026-02-12 14:11:42,197 - INFO - Loaded SOM boundaries. Rows: 118
2026-02-12 14:11:42,286 - INFO - [ETH] Spatially joined Admin1 info.
2026-02-12 14:11:42,286 - INFO - Loaded ETH boundaries. Rows: 74


Loaded 482 total regions.
      shapeName shapeISO                 shapeID shapeGroup shapeType  \
0      Ainabkoi      KEN  3690345B95414198105650        KEN      ADM2   
1       Ainamoi      KEN  3690345B54801368411234        KEN      ADM2   
2         Aldai      KEN  3690345B36924206491537        KEN      ADM2   
3  Alego Usonga      KEN  3690345B57306569752992        KEN      ADM2   
4        Awendo      KEN  3690345B13715496647196        KEN      ADM2   

                                            geometry       admin1  
0  POLYGON ((35.4633 0.5072, 35.46246 0.50691, 35...  Uasin Gishu  
1  POLYGON ((35.32495 -0.25056, 35.3225 -0.24841,...      Kericho  
2  POLYGON ((35.13034 0.1363, 35.12748 0.13635, 3...       Kisumu  
3  POLYGON ((34.21848 0.16479, 34.21677 0.1634, 3...        Siaya  
4  POLYGON ((34.62157 -0.9854, 34.61932 -0.9833, ...       Migori  


## 4. Process Price Data

In [ ]:
logger.info("Standardizing Price Data via Spatial Join...")
if 'lat' in price_df.columns and 'lon' in price_df.columns:
    price_joined = spatial_join_points(price_df, admin_gdf, 'lon', 'lat')
    matched = price_joined['admin2_canonical'].notna().sum()
    total = len(price_joined)
    logger.info(f"Price spatial join: {matched}/{total} matched ({matched/total*100:.1f}%)")
else:
    logger.warning("Price data has no lat/lon columns. Falling back to fuzzy matching.")
    price_mapping = fuzzy_match_names(price_df['adm2_name'], canonical_names)
    price_joined = price_df.copy()
    price_joined['admin2_canonical'] = price_joined['adm2_name'].map(price_mapping)


# KEEP ALL COMMODITIES that are present
# KEEP_COMMODITIES = ['sorghum', 'maize', 'rice', 'wheat', 'beans', 'millet', 'food_price_index']
# price_cols = [c for c in price_joined.columns
#                 if any(c == k or c.startswith(k + '_') or c.endswith('_' + k)
#                         or c.startswith('l_' + k) or c.startswith('o_' + k)
#                         or c.startswith('inflation_' + k) or c.startswith('trust_' + k)
#                         for k in KEEP_COMMODITIES)]
price_cols=['c_maize']
# Keep only numeric among selected
price_cols = price_joined[price_cols].select_dtypes(include=np.number).columns.tolist()
logger.info(f"Price columns kept: {len(price_cols)}")

price_agg = price_joined.groupby(
    ['year', 'month', 'shapeISO', 'admin1', 'admin2_canonical']
)[price_cols].mean().reset_index().rename(columns={'shapeISO': 'country_iso'})

print("Price Aggregation Sample:")
print(price_agg.head())
print(len(country_iso))



2026-02-12 14:13:56,974 - INFO - Standardizing Price Data via Spatial Join...
2026-02-12 14:13:56,976 - WARNING - Dropping 460 rows with missing coordinates (NaN in lon or lat).
2026-02-12 14:13:57,622 - INFO - 460 points not inside any polygon (e.g. coastal/border). Attempting nearest match...
/Users/halimjun/Coding_local/price_prediction_clean/venv/lib/python3.11/site-packages/geopandas/geodataframe.py:1891: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  result = super().__getitem__(key)
/Users/halimjun/Coding_local/price_prediction_clean/venv/lib/python3.11/site-packages/geopandas/array.py:407: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
2026-02-12 14:13:57,764 - INFO - Nearest match recovered 460 points. Still unmatched: 0
2026-02-12 14:13:58,004 - INFO - Price spatial join: 62330/62330 matc

Price Aggregation Sample:
   year  month country_iso   admin1 admin2_canonical      c_maize
0  2007      1         ETH   Somali            Afder  2358.090000
1  2007      1         ETH   Somali         Shabelle  2513.400000
2  2007      1         KEN  Garissa           Dadaab    18.184286
3  2007      1         KEN  Garissa          Lagdera    19.782500
4  2007      1         KEN  Garissa      Wajir South    19.210000
25990


In [289]:
price_joined[price_joined['shapeISO']=='ETH']

,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,DATES,year,...,c_food_price_index,inflation_food_price_index,trust_food_price_index,index_right,admin2_canonical,shapeISO,shapeID,shapeGroup,shapeType,admin1
543396,SOM,Somalia,Gedo,Doolow,Doolow,4.16,42.08,gid_41600000420800000,2007-01-01,2007,...,0.25,NaN,9.3,408.0,Afder,ETH,57463737B43250517747981,ETH,ADM2,Somali
543397,SOM,Somalia,Gedo,Doolow,Doolow,4.16,42.08,gid_41600000420800000,2007-02-01,2007,...,0.24,NaN,9.3,408.0,Afder,ETH,57463737B43250517747981,ETH,ADM2,Somali
543398,SOM,Somalia,Gedo,Doolow,Doolow,4.16,42.08,gid_41600000420800000,2007-03-01,2007,...,0.27,NaN,9.3,408.0,Afder,ETH,57463737B43250517747981,ETH,ADM2,Somali
543399,SOM,Somalia,Gedo,Doolow,Doolow,4.16,42.08,gid_41600000420800000,2007-04-01,2007,...,0.27,NaN,9.3,408.0,Afder,ETH,57463737B43250517747981,ETH,ADM2,Somali
543400,SOM,Somalia,Gedo,Doolow,Doolow,4.16,42.08,gid_41600000420800000,2007-05-01,2007,...,0.27,NaN,9.3,408.0,Afder,ETH,57463737B43250517747981,ETH,ADM2,Somali
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547991,SOM,Somalia,Wadajir,Wadajir,Wadajir,5.00,45.00,gid_50000000450000000,2025-10-01,2025,...,1.44,-4.59,9.3,458.0,Shabelle,ETH,57463737B16105117004736,ETH,ADM2,Somali
547992,SOM,Somalia,Wadajir,Wadajir,Wadajir,5.00,45.00,gid_50000000450000000,2025-11-01,2025,...,1.44,-3.84,9.3,458.0,Shabelle,ETH,57463737B16105117004736,ETH,ADM2,Somali
547993,SOM,Somalia,Wadajir,Wadajir,Wadajir,5.00,45.00,gid_50000000450000000,2025-12-01,2025,...,1.47,-1.50,9.3,458.0,Shabelle,ETH,57463737B16105117004736,ETH,ADM2,Somali
547994,SOM,Somalia,Wadajir,Wadajir,Wadajir,5.00,45.00,gid_50000000450000000,2026-01-01,2026,...,1.46,3.14,9.3,458.0,Shabelle,ETH,57463737B16105117004736,ETH,ADM2,Somali


## 5. Process Population Data

In [242]:
logger.info("Standardizing Population Data via WorldPop Zonal Stats...")
pop_dfs = []

# Map ISO to raster file
raster_map = {
    'KEN': 'ken_pop_2020_1km.tif',
    'SOM': 'som_pop_2020_1km.tif',
    'ETH': 'eth_pop_2020_1km.tif'
}

for iso in iso3_list:
    if iso not in raster_map:
        continue
        
    raster_path = Path(DATA_DIR) / 'population_worldpop' / raster_map[iso]
    
    if raster_path.exists():
        iso_pop = process_worldpop_population(admin_gdf, str(raster_path), iso)
        if not iso_pop.empty:
            pop_dfs.append(iso_pop)
    else:
        logger.warning(f"Population raster not found for {iso}: {raster_path}")

if pop_dfs:
    pop_agg = pd.concat(pop_dfs, ignore_index=True)
    pop_agg = pop_agg.groupby(['shapeISO', 'admin1', 'admin2_canonical'])['population'].sum().reset_index()
else:
    logger.warning("No population data processed.")
    pop_agg = pd.DataFrame(columns=['shapeISO', 'admin1', 'admin2_canonical', 'population'])

print("Population Aggregation Sample:")
pop_agg.head()



2026-02-12 14:14:07,177 - INFO - Standardizing Population Data via WorldPop Zonal Stats...
2026-02-12 14:14:07,179 - INFO - Processing WorldPop for KEN with 482 regions...
2026-02-12 14:14:08,152 - INFO - Processing WorldPop for SOM with 482 regions...
2026-02-12 14:14:08,409 - INFO - Processing WorldPop for ETH with 482 regions...


Population Aggregation Sample:


,shapeISO,admin1,admin2_canonical,population
0,ETH,Addis Ababa,East Shewa,2423843.000
1,ETH,Addis Ababa,North Shewa(R4),1804054.500
2,ETH,Addis Ababa,Region 14,4118307.750
3,ETH,Addis Ababa,South West Shewa,1865568.625
4,ETH,Addis Ababa,West Shewa,3454740.000


## 6. Process Crop Data

In [243]:
logger.info("Standardizing Crop Data...")
crop_df_proc = crop_df.copy()
# Filter to target countries
if 'shapeISO_ADM0' in crop_df_proc.columns:
    crop_df_proc = crop_df_proc[crop_df_proc['shapeISO_ADM0'].isin(iso3_list)]
    logger.info(f"Crop data filtered to {iso3_list}: {len(crop_df_proc)} rows")

canonical_set = set(canonical_names)
crop_df_proc['admin2_canonical'] = crop_df_proc['shapeName_ADM2'].where(
    crop_df_proc['shapeName_ADM2'].isin(canonical_set)
)
unmatched_crop = crop_df_proc['admin2_canonical'].isna().sum()
if unmatched_crop > 0:
    logger.warning(f"Crop data: {unmatched_crop} rows with unmatched Admin2 names. Falling back to fuzzy match.")
    remaining_crop = crop_df_proc.loc[crop_df_proc['admin2_canonical'].isna(), 'shapeName_ADM2']
    crop_fallback = fuzzy_match_names(remaining_crop, canonical_names)
    crop_df_proc.loc[crop_df_proc['admin2_canonical'].isna(), 'admin2_canonical'] = remaining_crop.map(crop_fallback)


# Merge Admin1 and ISO Info
admin_info = admin_gdf[['shapeName', 'shapeISO', 'admin1']].drop_duplicates().rename(columns={'shapeName': 'admin2_canonical', 'shapeISO': 'country_iso'})
crop_df_proc = pd.merge(crop_df_proc, admin_info, on='admin2_canonical', how='left')

crop_agg = crop_df_proc.groupby(
    ['country_iso', 'admin1', 'admin2_canonical']
)['value'].mean().reset_index().rename(columns={'value': 'crop_cover_fraction'})

print("Crop Aggregation Sample:")
print(crop_agg.head())


2026-02-12 14:14:15,334 - INFO - Standardizing Crop Data...
2026-02-12 14:14:15,336 - INFO - Crop data filtered to ['KEN', 'SOM', 'ETH']: 454 rows


Crop Aggregation Sample:
  country_iso       admin1  admin2_canonical  crop_cover_fraction
0         ETH  Addis Ababa        East Shewa            71.789031
1         ETH  Addis Ababa   North Shewa(R4)            56.956141
2         ETH  Addis Ababa         Region 14            37.385937
3         ETH  Addis Ababa  South West Shewa            70.810667
4         ETH  Addis Ababa        West Shewa            51.487673


## 7. Process ACLED Data

In [228]:
acled_df.head()

,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
70832,1997-10-04,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,3,NaN,Political violence,896.0,8.9644,38.7756
70833,2001-11-24,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,1,NaN,Political violence,896.0,8.9644,38.7756
70834,2002-04-06,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,100,NaN,Political violence,896.0,8.9644,38.7756
70835,2011-03-19,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,0,NaN,Political violence,896.0,8.9644,38.7756
70836,2011-09-10,Eastern Africa,Ethiopia,Addis Ababa,Battles,Armed clash,1,4,NaN,Political violence,896.0,8.9644,38.7756


In [244]:
logger.info("Joining ACLED Data via Spatial Join...")
acled_joined = spatial_join_points(acled_df, admin_gdf, 'CENTROID_LONGITUDE', 'CENTROID_LATITUDE')

acled_joined['year'] = pd.to_datetime(acled_joined['WEEK']).dt.year
acled_joined['month'] = pd.to_datetime(acled_joined['WEEK']).dt.month

acled_agg = acled_joined.groupby(
    ['year', 'month', 'shapeISO', 'admin1', 'admin2_canonical']
).agg({
    'FATALITIES': 'sum',
    'EVENTS': 'count'
}).reset_index().rename(columns={'EVENTS': 'conflict_events', 'FATALITIES': 'conflict_fatalities', 'shapeISO': 'country_iso'})

print("ACLED Aggregation Sample:")
print(acled_agg.head())


2026-02-12 14:14:19,965 - INFO - Joining ACLED Data via Spatial Join...
2026-02-12 14:14:20,156 - INFO - 14 points not inside any polygon (e.g. coastal/border). Attempting nearest match...
/Users/halimjun/Coding_local/price_prediction_clean/venv/lib/python3.11/site-packages/geopandas/array.py:407: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
2026-02-12 14:14:20,192 - INFO - Nearest match recovered 0 points. Still unmatched: 14


ACLED Aggregation Sample:
   year  month country_iso      admin1  admin2_canonical  conflict_fatalities  \
0  1997      1         ETH       SNNPR            Agnuak                    0   
1  1997      1         KEN     Mombasa             Jomvu                    2   
2  1997      1         KEN      Nakuru  Nakuru Town East                    1   
3  1997      1         KEN  West Pokot             Loima                    3   
4  1997      2         ETH      Hareri            Hareri                    2   

   conflict_events  
0                1  
1                1  
2                2  
3                1  
4                1  


In [245]:
spi_df = process_spi_data(admin_gdf, DATA_DIR, start_year=2007)


2026-02-12 14:14:22,914 - INFO - Loaded SPI NetCDF. Filtering start_year >= 2007
2026-02-12 14:14:22,920 - INFO - Processing 408 time slices for SPI...
2026-02-12 14:14:22,921 - INFO - Processing SPI for 2007-1...


KeyboardInterrupt: 

In [246]:
spi_df

,year,month,country_iso,admin1,admin2_canonical,spi_6month
0,2007,1,KEN,Uasin Gishu,Ainabkoi,1.476436
1,2007,1,KEN,Kericho,Ainamoi,1.310457
2,2007,1,KEN,Kisumu,Aldai,1.029119
3,2007,1,KEN,Siaya,Alego Usonga,2.130918
4,2007,1,KEN,Migori,Awendo,1.501298
...,...,...,...,...,...,...
104107,2024,12,ETH,Somali,Zone 1,0.641804
104108,2024,12,ETH,Afar,Zone 2,1.145679
104109,2024,12,ETH,Somali,Zone 3,0.802517
104110,2024,12,ETH,Afar,Zone 4,1.034978


## 8. Final Merge

In [247]:
pop_agg.rename(columns={'shapeISO': 'country_iso'}, inplace=True)
spi_df

,year,month,country_iso,admin1,admin2_canonical,spi_6month
0,2007,1,KEN,Uasin Gishu,Ainabkoi,1.476436
1,2007,1,KEN,Kericho,Ainamoi,1.310457
2,2007,1,KEN,Kisumu,Aldai,1.029119
3,2007,1,KEN,Siaya,Alego Usonga,2.130918
4,2007,1,KEN,Migori,Awendo,1.501298
...,...,...,...,...,...,...
104107,2024,12,ETH,Somali,Zone 1,0.641804
104108,2024,12,ETH,Afar,Zone 2,1.145679
104109,2024,12,ETH,Somali,Zone 3,0.802517
104110,2024,12,ETH,Afar,Zone 4,1.034978


In [248]:
price_agg

,year,month,country_iso,admin1,admin2_canonical,c_maize
0,2007,1,ETH,Somali,Afder,2358.090000
1,2007,1,ETH,Somali,Shabelle,2513.400000
2,2007,1,KEN,Garissa,Dadaab,18.184286
3,2007,1,KEN,Garissa,Lagdera,19.782500
4,2007,1,KEN,Garissa,Wajir South,19.210000
...,...,...,...,...,...,...
25985,2026,2,SOM,Sool,BURAO,15319.720000
25986,2026,2,SOM,Sool,HUDUN,33046.160000
25987,2026,2,SOM,Woqooyi Galbeed,BERBERA,13959.960000
25988,2026,2,SOM,Woqooyi Galbeed,BORAMA,9012.990000


In [249]:
price_df.head()

,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,DATES,year,...,l_yogurt,c_yogurt,inflation_yogurt,trust_yogurt,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index
220746,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-01-01,2007,...,NaN,NaN,NaN,NaN,0.56,0.58,0.55,0.57,NaN,9.8
220747,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-02-01,2007,...,NaN,NaN,NaN,NaN,0.57,0.58,0.55,0.56,NaN,9.8
220748,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-03-01,2007,...,NaN,NaN,NaN,NaN,0.56,0.57,0.54,0.54,NaN,9.8
220749,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-04-01,2007,...,NaN,NaN,NaN,NaN,0.54,0.56,0.53,0.56,NaN,9.8
220750,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-05-01,2007,...,NaN,NaN,NaN,NaN,0.56,0.58,0.55,0.56,NaN,9.8


In [251]:
price_joined.head()

,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,DATES,year,...,c_food_price_index,inflation_food_price_index,trust_food_price_index,index_right,admin2_canonical,shapeISO,shapeID,shapeGroup,shapeType,admin1
220746,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-01-01,2007,...,0.57,NaN,9.8,22.0,Bura,KEN,3690345B15022807723985,KEN,ADM2,Kitui
220747,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-02-01,2007,...,0.56,NaN,9.8,22.0,Bura,KEN,3690345B15022807723985,KEN,ADM2,Kitui
220748,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-03-01,2007,...,0.54,NaN,9.8,22.0,Bura,KEN,3690345B15022807723985,KEN,ADM2,Kitui
220749,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-04-01,2007,...,0.56,NaN,9.8,22.0,Bura,KEN,3690345B15022807723985,KEN,ADM2,Kitui
220750,KEN,Kenya,Coast,Tana River,Adele Center,-0.47,39.61,gid_-4700000396100000,2007-05-01,2007,...,0.56,NaN,9.8,22.0,Bura,KEN,3690345B15022807723985,KEN,ADM2,Kitui


In [250]:
years = sorted(price_df['year'].unique())
months = sorted(price_df['month'].unique())

logger.info(f"Creating skeleton for {len(years)} years, {len(months)} months, {len(canonical_names)} regions...")
master = create_master_skeleton(years, months, admin_gdf)
master.head()

2026-02-12 14:14:48,979 - INFO - Creating skeleton for 20 years, 12 months, 482 regions...


,admin2,country_iso,admin1,year,month
0,Ainabkoi,KEN,Uasin Gishu,2007,1
1,Ainamoi,KEN,Kericho,2007,1
2,Aldai,KEN,Kisumu,2007,1
3,Alego Usonga,KEN,Siaya,2007,1
4,Awendo,KEN,Migori,2007,1


In [270]:
acled_agg

,year,month,country_iso,admin1,admin2_canonical,conflict_fatalities,conflict_events
0,1997,1,ETH,SNNPR,Agnuak,0,1
1,1997,1,KEN,Mombasa,Jomvu,2,1
2,1997,1,KEN,Nakuru,Nakuru Town East,1,2
3,1997,1,KEN,West Pokot,Loima,3,1
4,1997,2,ETH,Hareri,Hareri,2,1
...,...,...,...,...,...,...,...
11747,2026,1,SOM,Lower Shebelle,AWDHEEGLE,9,6
11748,2026,1,SOM,Lower Shebelle,BURHAKABA,27,6
11749,2026,1,SOM,Lower Shebelle,JOWHAR,15,3
11750,2026,1,SOM,Woqooyi Galbeed,HARGEISA,0,1


In [272]:

# logger.info("Merging all datasets...")

# merged = pd.merge(master, price_agg,
#                     left_on=['year', 'month', 'country_iso', 'admin1', 'admin2'],
#                     right_on=['year', 'month', 'country_iso', 'admin1', 'admin2_canonical'], how='left')
# if 'admin2_canonical' in merged.columns:
#     merged.drop(columns=['admin2_canonical'], inplace=True)

merged = pd.merge(price_agg, pop_agg,
                    left_on=['country_iso', 'admin1', 'admin2_canonical'], right_on=['country_iso', 'admin1', 'admin2_canonical'], how='left')
# if 'admin2_canonical' in merged.columns:
#     merged.drop(columns=['admin2_canonical'], inplace=True)
print(merged.head())
merged = pd.merge(merged, crop_agg,
                    left_on=['country_iso', 'admin1', 'admin2_canonical'], right_on=['country_iso', 'admin1', 'admin2_canonical'], how='left')
# if 'admin2_canonical' in merged.columns:
#     merged.drop(columns=['admin2_canonical'], inplace=True)

merged = pd.merge(merged, acled_agg,
                    left_on=['year', 'month', 'country_iso', 'admin1', 'admin2_canonical'],
                    right_on=['year', 'month', 'country_iso', 'admin1', 'admin2_canonical'], how='left')
# if 'admin2_canonical' in merged.columns:
#     merged.drop(columns=['admin2_canonical'], inplace=True)

merged['conflict_events'] = merged['conflict_events'].fillna(0)
merged['conflict_fatalities'] = merged['conflict_fatalities'].fillna(0)

# logger.info(f"Final merged shape: {merged.shape}")

# Merge Macro Data (Global/Timeseries)
if not macro_df.empty:
    merged = pd.merge(merged, macro_df, on=['year', 'month'], how='left')
    logger.info(f"Merged Macro data. Shape: {merged.shape}")
merged

# Merge Night Lights (Yearly)
if 'night_lights_df' in locals() and not night_lights_df.empty:
    # Rename admin2 to match master if needed, or just merge on admin2
    # Master has 'admin2' (from admin_gdf shapeName). 
    # Night lights has 'admin2' (from admin_gdf shapeName).
    
    # We merge on country, admin2, year
    merged = pd.merge(merged, night_lights_df[['country_iso', 'admin2', 'year', 'night_light_mean']],
                     on=['country_iso', 'admin2', 'year'], 
                     how='left')
    logger.info(f"Merged Night Lights data. Shape: {merged.shape}")


2026-02-12 14:22:50,367 - INFO - Merged Macro data. Shape: (25990, 13)


   year  month country_iso   admin1 admin2_canonical      c_maize  \
0  2007      1         ETH   Somali            Afder  2358.090000   
1  2007      1         ETH   Somali         Shabelle  2513.400000   
2  2007      1         KEN  Garissa           Dadaab    18.184286   
3  2007      1         KEN  Garissa          Lagdera    19.782500   
4  2007      1         KEN  Garissa      Wajir South    19.210000   

     population  
0  9.639658e+05  
1  1.129154e+06  
2  2.844247e+05  
3  1.761893e+05  
4  3.591731e+05  


,year,month,country_iso,admin1,admin2_canonical,c_maize,population,crop_cover_fraction,conflict_fatalities,conflict_events,energy_index,food_index,fertilizer_index
0,2007,1,ETH,Somali,Afder,2358.090000,9.639658e+05,14.235164,0.0,0.0,73.608133,73.527464,61.076449
1,2007,1,ETH,Somali,Shabelle,2513.400000,1.129154e+06,17.473684,0.0,0.0,73.608133,73.527464,61.076449
2,2007,1,KEN,Garissa,Dadaab,18.184286,2.844247e+05,3.291667,0.0,0.0,73.608133,73.527464,61.076449
3,2007,1,KEN,Garissa,Lagdera,19.782500,1.761893e+05,3.857143,0.0,0.0,73.608133,73.527464,61.076449
4,2007,1,KEN,Garissa,Wajir South,19.210000,3.591731e+05,3.626459,0.0,0.0,73.608133,73.527464,61.076449
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25985,2026,2,SOM,Sool,BURAO,15319.720000,4.294055e+05,7.311111,0.0,0.0,NaN,NaN,NaN
25986,2026,2,SOM,Sool,HUDUN,33046.160000,5.668966e+04,NaN,0.0,0.0,NaN,NaN,NaN
25987,2026,2,SOM,Woqooyi Galbeed,BERBERA,13959.960000,1.990188e+05,1.333333,0.0,0.0,NaN,NaN,NaN
25988,2026,2,SOM,Woqooyi Galbeed,BORAMA,9012.990000,3.489901e+05,18.969295,0.0,0.0,NaN,NaN,NaN


In [274]:

# Merge SPI Data
if 'spi_df' in locals() and not spi_df.empty:
    merged = pd.merge(merged, spi_df, 
                     left_on=['year', 'month', 'country_iso', 'admin1', 'admin2_canonical'], 
                     right_on = ['year', 'month', 'country_iso', 'admin1', 'admin2_canonical'],
                     how='left')
    logger.info(f"Merged SPI data. Shape: {merged.shape}")

merged.head()


2026-02-12 14:23:26,410 - INFO - Merged SPI data. Shape: (25990, 14)


,year,month,country_iso,admin1,admin2_canonical,c_maize,population,crop_cover_fraction,conflict_fatalities,conflict_events,energy_index,food_index,fertilizer_index,spi_6month
0,2007,1,ETH,Somali,Afder,2358.090000,9.639658e+05,14.235164,0.0,0.0,73.608133,73.527464,61.076449,1.472507
1,2007,1,ETH,Somali,Shabelle,2513.400000,1.129154e+06,17.473684,0.0,0.0,73.608133,73.527464,61.076449,1.447565
2,2007,1,KEN,Garissa,Dadaab,18.184286,2.844247e+05,3.291667,0.0,0.0,73.608133,73.527464,61.076449,1.247316
3,2007,1,KEN,Garissa,Lagdera,19.782500,1.761893e+05,3.857143,0.0,0.0,73.608133,73.527464,61.076449,1.005821
4,2007,1,KEN,Garissa,Wajir South,19.210000,3.591731e+05,3.626459,0.0,0.0,73.608133,73.527464,61.076449,1.114754


In [277]:
merged[merged['country_iso']=='ETH']

,year,month,country_iso,admin1,admin2_canonical,c_maize,population,crop_cover_fraction,conflict_fatalities,conflict_events,energy_index,food_index,fertilizer_index,spi_6month
0,2007,1,ETH,Somali,Afder,2358.09,9.639658e+05,14.235164,0.0,0.0,73.608133,73.527464,61.076449,1.472507
1,2007,1,ETH,Somali,Shabelle,2513.40,1.129154e+06,17.473684,0.0,0.0,73.608133,73.527464,61.076449,1.447565
113,2007,2,ETH,Somali,Afder,2176.82,9.639658e+05,14.235164,0.0,0.0,79.140582,75.938259,63.935508,1.464269
114,2007,2,ETH,Somali,Shabelle,2417.86,1.129154e+06,17.473684,0.0,0.0,79.140582,75.938259,63.935508,1.441375
226,2007,3,ETH,Somali,Afder,2066.85,9.639658e+05,14.235164,0.0,0.0,82.705339,75.638769,64.268397,1.288941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25652,2025,12,ETH,Somali,Shabelle,20058.60,1.129154e+06,17.473684,0.0,0.0,NaN,NaN,NaN,NaN
25764,2026,1,ETH,Somali,Afder,23428.92,9.639658e+05,14.235164,0.0,0.0,NaN,NaN,NaN,NaN
25765,2026,1,ETH,Somali,Shabelle,19828.54,1.129154e+06,17.473684,0.0,0.0,NaN,NaN,NaN,NaN
25877,2026,2,ETH,Somali,Afder,24081.96,9.639658e+05,14.235164,0.0,0.0,NaN,NaN,NaN,NaN


In [280]:
price_df[price_df['ISO3']=='ETH']

,ISO3,country,adm1_name,adm2_name,mkt_name,lat,lon,geo_id,DATES,year,...,l_yogurt,c_yogurt,inflation_yogurt,trust_yogurt,o_food_price_index,h_food_price_index,l_food_price_index,c_food_price_index,inflation_food_price_index,trust_food_price_index
